In [1]:
from pathlib import Path
from isolate_parser import Isolate
from isolate_parser.tools import helper_functions as hf
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re

In [ ]:
TSV_PATH: Path = Path("/dbcan-annotations/CAZyme-annotations")

In [ ]:
taxonomic_classification_csv = Path("DOME2-Isolates-Taxonomy.csv")

In [4]:
culturomics_csv = pd.read_csv("Culturomics-Metadata.csv")

In [5]:
EXAMPLE_TSV = pd.read_csv(f"{TSV_PATH}/DOME2_0672/overview.tsv", sep="\t")

In [6]:
EXAMPLE_TSV.columns

Index(['Gene ID', 'EC#', 'dbCAN_hmm', 'dbCAN_sub', 'DIAMOND', '#ofTools',
       'Recommend Results'],
      dtype='object')

In [7]:
culturomics_csv.head()

,Sample Name,DomeNumber
0,19-W0326-V303ns01,DOME2_0001
1,19-W0502-V402ns01,DOME2_0002
2,19-W0502-V402ns01,DOME2_0003
3,19-D0302-V303ns01,DOME2_0004
4,19-D0302-V303ns01,DOME2_0005


In [8]:
search_queries = [
    culturomics_csv["Sample Name"].str.contains("C", case=False),
    culturomics_csv["Sample Name"].str.contains("W", case=False),
    culturomics_csv["Sample Name"].str.contains("D", case=False),
]
cohort_type = ["Cow", "Farmer", "Office Worker"]
culturomics_csv["Subject Type"] = np.select(
    search_queries, cohort_type, default="Unknown"
)

In [9]:
culturomics_csv.head()

,Sample Name,DomeNumber,Subject Type
0,19-W0326-V303ns01,DOME2_0001,Farmer
1,19-W0502-V402ns01,DOME2_0002,Farmer
2,19-W0502-V402ns01,DOME2_0003,Farmer
3,19-D0302-V303ns01,DOME2_0004,Office Worker
4,19-D0302-V303ns01,DOME2_0005,Office Worker


In [10]:
msa_isolates = pd.read_csv("msa-isolates.csv")

In [11]:
mucin_isolates = pd.read_csv("mucin-isolates.csv")

In [12]:
msa_isolates.head()

,DomeNumber,Media Type
0,DOME2_0259,MSA
1,DOME2_0010,MSA
2,DOME2_0051,MSA
3,DOME2_0264,MSA
4,DOME2_0031,MSA


In [13]:
len(msa_isolates)

454

In [14]:
mucin_isolates

,DomeNumber,Media Type
0,DOME2_0171,Mucin
1,DOME2_0150,Mucin
2,DOME2_0173,Mucin
3,DOME2_0145,Mucin
4,DOME2_0142,Mucin
...,...,...
150,DOME2_0166,Mucin
151,DOME2_0137,Mucin
152,DOME2_0598,Mucin
153,DOME2_0162,Mucin


In [15]:
len(mucin_isolates)

155

In [16]:
taxonomic_classification = pd.read_csv(taxonomic_classification_csv)

In [17]:
taxonomic_classification.head()

,user_genome,classification
0,DOME2_0001,staphylococcus epidermidis
1,DOME2_0002,mammaliicoccus vitulinus
2,DOME2_0003,bacillus paralicheniformis
3,DOME2_0004,bacillus licheniformis
4,DOME2_0005,staphylococcus epidermidis


In [18]:
taxonomic_classification = taxonomic_classification.rename(
    columns={"user_genome": "DomeNumber"}
)

In [19]:
taxonomic_classification.head()

,DomeNumber,classification
0,DOME2_0001,staphylococcus epidermidis
1,DOME2_0002,mammaliicoccus vitulinus
2,DOME2_0003,bacillus paralicheniformis
3,DOME2_0004,bacillus licheniformis
4,DOME2_0005,staphylococcus epidermidis


In [20]:
merged_df = pd.merge(culturomics_csv, mucin_isolates, on="DomeNumber", how="left")
merged_df = pd.merge(merged_df, msa_isolates, on="DomeNumber", how="left")
merged_df = pd.merge(merged_df, taxonomic_classification, on="DomeNumber", how="left")

In [21]:
merged_df.head()

,Sample Name,DomeNumber,Subject Type,Media Type_x,Media Type_y,classification
0,19-W0326-V303ns01,DOME2_0001,Farmer,NaN,MSA,staphylococcus epidermidis
1,19-W0502-V402ns01,DOME2_0002,Farmer,NaN,MSA,mammaliicoccus vitulinus
2,19-W0502-V402ns01,DOME2_0003,Farmer,NaN,MSA,bacillus paralicheniformis
3,19-D0302-V303ns01,DOME2_0004,Office Worker,NaN,MSA,bacillus licheniformis
4,19-D0302-V303ns01,DOME2_0005,Office Worker,NaN,MSA,staphylococcus epidermidis


In [22]:
merged_df["Media"] = merged_df["Media Type_x"].combine_first(merged_df["Media Type_y"])
merged_df = merged_df.drop(columns=["Media Type_x", "Media Type_y"])
culturomics_metadata = merged_df.dropna(subset=["Media"])

In [23]:
culturomics_metadata.head()

,Sample Name,DomeNumber,Subject Type,classification,Media
0,19-W0326-V303ns01,DOME2_0001,Farmer,staphylococcus epidermidis,MSA
1,19-W0502-V402ns01,DOME2_0002,Farmer,mammaliicoccus vitulinus,MSA
2,19-W0502-V402ns01,DOME2_0003,Farmer,bacillus paralicheniformis,MSA
3,19-D0302-V303ns01,DOME2_0004,Office Worker,bacillus licheniformis,MSA
4,19-D0302-V303ns01,DOME2_0005,Office Worker,staphylococcus epidermidis,MSA


In [24]:
len(culturomics_metadata)

609

In [25]:
final_dome_isolate_list: list = (
    Path("Final-DOME2-Isolate-Passed-QC.sorted.txt").read_text().splitlines()
)

In [26]:
final_dome_isolate_list[3]

'DOME2_0004'

In [27]:
len(final_dome_isolate_list)

582

In [28]:
filtered_culturomics_metadata = culturomics_metadata.query(
    "DomeNumber in @final_dome_isolate_list"
)

In [29]:
filtered_culturomics_metadata.head()

,Sample Name,DomeNumber,Subject Type,classification,Media
0,19-W0326-V303ns01,DOME2_0001,Farmer,staphylococcus epidermidis,MSA
1,19-W0502-V402ns01,DOME2_0002,Farmer,mammaliicoccus vitulinus,MSA
2,19-W0502-V402ns01,DOME2_0003,Farmer,bacillus paralicheniformis,MSA
3,19-D0302-V303ns01,DOME2_0004,Office Worker,bacillus licheniformis,MSA
4,19-D0302-V303ns01,DOME2_0005,Office Worker,staphylococcus epidermidis,MSA


In [30]:
len(filtered_culturomics_metadata)

582

In [31]:
EXAMPLE_TSV.head()

,Gene ID,EC#,dbCAN_hmm,dbCAN_sub,DIAMOND,#ofTools,Recommend Results
0,10_11,3.2.1.20:2,GH13_30(46-414),GH13_e234(45-416),GH13_30,3,GH13_30
1,10_2,-,AA7(31-204),-,-,1,-
2,10_80,-,-,-,AA1,1,-
3,10_81,-,-,-,GH75,1,-
4,11_42,3.5.1.115:3,CE14(10-145),CE14_e1(11-146),CE14,3,CE14_e1


In [32]:
EXAMPLE_TSV = EXAMPLE_TSV.query("'-' not in `Recommend Results`")

In [33]:
EXAMPLE_TSV.head()

,Gene ID,EC#,dbCAN_hmm,dbCAN_sub,DIAMOND,#ofTools,Recommend Results
0,10_11,3.2.1.20:2,GH13_30(46-414),GH13_e234(45-416),GH13_30,3,GH13_30
4,11_42,3.5.1.115:3,CE14(10-145),CE14_e1(11-146),CE14,3,CE14_e1
5,11_55,1.1.3.-:1,AA7(76-505),AA7_e0(80-293),-,2,AA7_e0
6,11_6,-,GT39(39-299),GT39_e21(40-300),GT39,3,GT39_e21
11,11_88,2.4.1.282:1|2.4.1.216:1,GH65(611-1021),GH65_e15(613-1022),GH65,3,GH65_e15


In [34]:
len(EXAMPLE_TSV["Recommend Results"])

88

In [35]:
example_CAZyme_gene_hits = EXAMPLE_TSV["Recommend Results"].to_list()

In [36]:
print(example_CAZyme_gene_hits)

['GH13_30', 'CE14_e1', 'AA7_e0', 'GT39_e21', 'GH65_e15', 'GT2', 'GH3_e66', 'GH2_e112', 'GH184', 'GT119', 'GT20_e1', 'AA7_e0', 'AA3_2', 'GT4_e582', 'GH25_e239', 'GT4_e464', 'GT4_e26', 'GT4_e3623', 'GT2', 'GT2', 'GT4_e1368|GT4_e3273', 'GT1_e155', 'CBM48_e2|CBM48_e2|GH13_9', 'GH13_16', 'GH13_3', 'GT35_e8', 'GH13_11', 'GH77_e2', 'GT4_e4', 'GT51_e64', 'GT2', 'GT2', 'GT2', 'GT2', 'CE14_e40', 'CBM48_e3|GH13_11', 'GH13_26', 'CBM48_e0|GH13_10', 'GT4_e3552', 'GH76_e12', 'GH13_4', 'AA1_e26', 'GH15_e59', 'GT2', 'GH15_e59', 'GT128', 'GT4_e1928', 'GT26_e9', 'GH184', 'GT4_e2498', 'GT2', 'GT2', 'GT4_e1800', 'GT94_e7', 'GT4_e1298', 'GT2', 'GT4_e3080|GT4_e3368', 'GT2', 'GT2', 'GT4_e3696', 'GT4_e3343', 'GH79_e19', 'GH33_e24', 'GH33_e2', 'GT4_e2913', 'AA3_e57', 'AA4_e1', 'CE9_e64', 'GH109_e7', 'GT4_e1897', 'GT2', 'GT2', 'CE14', 'GT2', 'CE14_e37', 'CE14_e39', 'GT2', 'GT2', 'GT4_e1823', 'GT76_e12', 'GT4_e730', 'AA4_e1', 'GT4_e3984', 'GT2', 'GT2', 'CBM50_e312|CBM50_e966|GH23_e699', 'GT119', 'GT28_e141']


In [37]:
EXAMPLE_TSV.query("'-' not in `DIAMOND`")

,Gene ID,EC#,dbCAN_hmm,dbCAN_sub,DIAMOND,#ofTools,Recommend Results
0,10_11,3.2.1.20:2,GH13_30(46-414),GH13_e234(45-416),GH13_30,3,GH13_30
4,11_42,3.5.1.115:3,CE14(10-145),CE14_e1(11-146),CE14,3,CE14_e1
6,11_6,-,GT39(39-299),GT39_e21(40-300),GT39,3,GT39_e21
11,11_88,2.4.1.282:1|2.4.1.216:1,GH65(611-1021),GH65_e15(613-1022),GH65,3,GH65_e15
13,12_50,-,GH3(144-403),GH3_e66(145-404),GH3,3,GH3_e66
...,...,...,...,...,...,...,...
164,9_112,-,-,GT2(6-223),GT0,2,GT2
166,9_26,-,GT2(11-163),GT2(12-163),GT2,3,GT2
168,9_68,-|-|-,GH23(266-375),CBM50_e312(65-107)+CBM50_e966(137-180)+GH23_e6...,GH23,3,CBM50_e312|CBM50_e966|GH23_e699
170,9_81,-,GT119(81-423),-,GT119,2,GT119


In [38]:
from collections import Counter

In [39]:
example_CAZyme_gene_hits_counter = Counter(example_CAZyme_gene_hits)

In [40]:
example_CAZyme_gene_hits_counter

Counter({'GT2': 20,
         'AA7_e0': 2,
         'GH184': 2,
         'GT119': 2,
         'GH15_e59': 2,
         'AA4_e1': 2,
         'GH13_30': 1,
         'CE14_e1': 1,
         'GT39_e21': 1,
         'GH65_e15': 1,
         'GH3_e66': 1,
         'GH2_e112': 1,
         'GT20_e1': 1,
         'AA3_2': 1,
         'GT4_e582': 1,
         'GH25_e239': 1,
         'GT4_e464': 1,
         'GT4_e26': 1,
         'GT4_e3623': 1,
         'GT4_e1368|GT4_e3273': 1,
         'GT1_e155': 1,
         'CBM48_e2|CBM48_e2|GH13_9': 1,
         'GH13_16': 1,
         'GH13_3': 1,
         'GT35_e8': 1,
         'GH13_11': 1,
         'GH77_e2': 1,
         'GT4_e4': 1,
         'GT51_e64': 1,
         'CE14_e40': 1,
         'CBM48_e3|GH13_11': 1,
         'GH13_26': 1,
         'CBM48_e0|GH13_10': 1,
         'GT4_e3552': 1,
         'GH76_e12': 1,
         'GH13_4': 1,
         'AA1_e26': 1,
         'GT128': 1,
         'GT4_e1928': 1,
         'GT26_e9': 1,
         'GT4_e2498': 1,
      

In [52]:
# Computational
# MUCIN_CAZYMES: list = ['GH33','GH16','GH29','GH95','GH20','GH2','GH35','GH42','GH98','GH101','GH129','GH89','GH85','GH84']
# Experimentally determined
# MUCIN_CAZYMES=["GH2","GH18","GH20","GH27","GH29","GH31","GH33","GH35","GH36","GH42","GH95","GH98","GH101","GH109","GH112","GH151"]
isolates: list = []
for directory in TSV_PATH.iterdir():
    if directory.is_dir() and "DOME2" in os.path.basename(directory):
        TSV_EXISTS: bool = any(directory.glob("*.tsv"))
        if TSV_EXISTS:
            try:
                isolate: Isolate = Isolate(os.path.basename(directory))
                dbcan_df = pd.read_csv(f"{directory}/overview.tsv", sep="\t")
                formatting_results_df = dbcan_df["DIAMOND"].to_frame()
                # results_df = formatting_results_df[
                #     formatting_results_df['DIAMOND'].isin(MUCIN_CAZYMES)
                # ]
                results_df = formatting_results_df.query("'-' not in `DIAMOND`")
                cazymes: dict = {
                    "cazymes": dict(Counter(results_df["DIAMOND"].to_list()))
                }
                isolate.append_isolate_metadata(cazymes)
                isolate_metadata_df = filtered_culturomics_metadata.query(
                    "@isolate.sample_name in DomeNumber"
                )
                subject_type = isolate_metadata_df["Subject Type"].iloc[0]
                media_type = isolate_metadata_df["Media"].iloc[0]
                classification = isolate_metadata_df["classification"].iloc[0]
                dome_number = isolate_metadata_df["DomeNumber"].iloc[0]
                nasal_sample = re.sub(
                    r"^19-", "", isolate_metadata_df["Sample Name"].iloc[0]
                ).upper()
                isolate_metadata: dict = {
                    "culturomics": {
                        "subject type": subject_type,
                        "media": media_type,
                        "classification": classification,
                        "dome number": dome_number,
                        "sample number": nasal_sample,
                    },
                }
                isolate.append_isolate_metadata(isolate_metadata)
                isolates.append(isolate)
            except FileNotFoundError:
                pass

In [53]:
len(isolates)

582

In [54]:
culturomics_samples: list = [
    sample for sample in filtered_culturomics_metadata["DomeNumber"].to_list()
]

In [55]:
len(culturomics_samples)

582

In [56]:
culturomics_samples[1:6]

['DOME2_0002', 'DOME2_0003', 'DOME2_0004', 'DOME2_0005', 'DOME2_0006']

In [57]:
cazyme_isolates: list = [isolate.sample_name for isolate in isolates]

In [58]:
len(cazyme_isolates)

582

In [59]:
difference = list(set(culturomics_samples) - set(cazyme_isolates))

In [60]:
difference

[]

In [61]:
isolate_total_cazyme_count_data: list = []
for isolate in isolates:
    cazyme_dict: dict = {isolate.sample_name: isolate.metadata["cazymes"]}
    isolate_total_cazyme_count_data.append(cazyme_dict)

In [62]:
len(isolate_total_cazyme_count_data)

582

In [63]:
cazyme_data = pd.DataFrame.from_dict(
    {
        key: value
        for sample in isolate_total_cazyme_count_data
        for key, value in sample.items()
    },
    orient="index",
)
cazyme_data = (
    cazyme_data.fillna(0)
    .astype(int)
    .reset_index()
    .rename(columns={"index": "Dome Number"})
)

In [64]:
cazyme_data.head()

,Dome Number,GH1,GH131,GH76,GT41,CE8,GT22,CBM50,GH13_29,GH13_31,...,CBM96+PL8,CEN27346.1+GH140,CE12+PL11_1,CEA09274.1+GH0,GH13_32,CBM41+GH13_12,GT4+GT97,GH43_24,GH30_4,GH43_30
0,DOME2_0590,6,1,1,1,1,2,10,1,6,...,0,0,0,0,0,0,0,0,0,0
1,DOME2_0491,6,1,0,1,2,2,8,1,5,...,0,0,0,0,0,0,0,0,0,0
2,DOME2_0119,4,1,0,1,1,1,9,0,5,...,0,0,0,0,0,0,0,0,0,0
3,DOME2_0057,5,1,0,1,1,2,10,1,6,...,0,0,0,0,0,0,0,0,0,0
4,DOME2_0538,5,1,0,1,1,2,10,1,6,...,0,0,0,0,0,0,0,0,0,0


In [65]:
cazyme_data.sort_values(by="Dome Number").reset_index(drop=True).head()

,Dome Number,GH1,GH131,GH76,GT41,CE8,GT22,CBM50,GH13_29,GH13_31,...,CBM96+PL8,CEN27346.1+GH140,CE12+PL11_1,CEA09274.1+GH0,GH13_32,CBM41+GH13_12,GT4+GT97,GH43_24,GH30_4,GH43_30
0,DOME2_0001,4,1,0,1,1,1,8,0,5,...,0,0,0,0,0,0,0,0,0,0
1,DOME2_0002,9,1,1,1,2,2,10,1,4,...,0,0,0,0,0,0,0,0,0,0
2,DOME2_0003,12,1,0,1,4,2,18,2,6,...,0,0,0,0,0,0,0,0,0,0
3,DOME2_0004,10,1,0,1,4,2,17,2,6,...,0,0,0,0,0,0,0,0,0,0
4,DOME2_0005,4,1,0,1,1,1,8,0,5,...,0,0,0,0,0,0,0,0,0,0


In [66]:
len(cazyme_data["Dome Number"])

582

In [71]:
cazyme_dataframe

,subject type,media,cazyme count,classification,dome number,Sample ID
0,Non-Farmer,MSA,1,staphylococcus aureus,DOME2_0590,D0102-V302NS01
1,Cow,MSA,4,staphylococcus equorum,DOME2_0491,C0336-V202NS01
2,Farmer,MSA,1,staphylococcus epidermidis,DOME2_0119,W0241-V402NS01
3,Farmer,MSA,1,staphylococcus aureus,DOME2_0057,W0222-V102NS01
4,Farmer,MSA,1,staphylococcus aureus,DOME2_0538,W0222-V202NS01
...,...,...,...,...,...,...
577,Non-Farmer,MSA,1,staphylococcus aureus,DOME2_0035,D0117-V303NS01
578,Cow,Mucin,9,bacillus licheniformis,DOME2_0601,C0129-V301NS01
579,Cow,MSA,4,mammaliicoccus vitulinus,DOME2_0526,C0234-V203NS01
580,Farmer,MSA,1,staphylococcus aureus,DOME2_0384,W0127-V302NS01


In [ ]:
cazyme_dataframe.to_csv("DOME-Isolates-Subject-Mucin-CAZyme-Count.csv", index=False)